# Surface geostrophic velocities from sea level

Surface geostrophic velocity from sea level:

$$u_g = -\frac{g}{f}\frac{\partial\eta}{\partial y},
\qquad v_g = \frac{g}{f}\frac{\partial\eta}{\partial x}$$

The original differences on MOM5's staggered B-grid with `xgcm`.
After `regrid`, `zos` is on a regular grid, so the derivatives are
central differences with spherical metrics evaluated at the
tracer points and no interpolation between grid positions is
needed.

**Frequency caveat.** CMIP6 publishes no daily `zos`: the `Oday`
table holds only `chlos`, `omldamax`, `phycos`, `sos`, `sossq`,
`tos` and `tossq`, and `zos` exists only in `Omon`. Monthly sea
level is fine for the time-mean geostrophic circulation, but the
velocity *variance* it gives is low-frequency, sea-level-driven
kinetic energy and **not** mesoscale EKE. The same gap means the
COSIMA `Eddy-Mean_Kinetic_Energy_Decomposition.ipynb` has no
faithful CMIP6 translation at all: it needs daily 3D velocities.

---

Translated from the COSIMA recipe
[`Geostrophic_Velocities_from_Sea_Level.ipynb`](https://github.com/COSIMA/cosima-recipes/blob/main/03-Advanced-Recipes/Geostrophic_Velocities_from_Sea_Level.ipynb)
onto CMORised CMIP6 data, using ESMValCore's `Dataset` and
preprocessors in place of the ACCESS-NRI intake catalog.

This notebook is standalone: it needs only `esmvalcore`,
`iris`, `numpy`, `matplotlib`, `cartopy` and `gsw`. The
equivalent ESMValTool recipe is
`recipes/recipe_cosima_*.yml`.

In [ ]:
import numpy as np
import iris
import iris.analysis.cartography
import iris.plot as iplt
import matplotlib.pyplot as plt

from esmvalcore.dataset import Dataset
import gsw
from esmvalcore.preprocessor import regrid

## Start a dask cluster

Every COSIMA recipe opens with one, and these need one for the same
reason, though it is worth being precise about *which* part it speeds up.

**What the cluster does accelerate.** Everything from `Dataset.load()`
through the ESMValCore preprocessors — reading multi-decade files,
`regrid`, `climate_statistics`, `extract_region` — is lazy and chunked.
That is the expensive part of these notebooks and it is what the workers
parallelise. Without a client, dask falls back to its threaded scheduler;
ESMValCore itself warns that the distributed scheduler is preferable.

**What it does not.** Once the preprocessed cube is reduced, the
diagnostic arithmetic realises it into a plain masked array and proceeds
eagerly in numpy. That is deliberate: contour tracing, per-column
remapping and density binning are iterative and index-based, so they do
not express well as array graphs and would be slower, not faster, spread
over workers. Realising *after* the time-mean is what keeps that
affordable — realising a 20-year daily 3-D field before reducing it would
not be.

So: size the cluster for the load-and-preprocess stage. One thread per
worker, as the COSIMA recipes use — with chunks this large, per-worker
memory binds well before thread count does.

On ARE, `Client(threads_per_worker=1)` picks up the cores you requested
for the session. Pass `n_workers=` and `memory_limit=` to be explicit.

**Running the equivalent recipe instead?** Do not start a client there.
ESMValTool manages dask itself, and it defaults to the threaded
scheduler; switch it on in your ESMValTool configuration:

```yaml
dask:
  use: local_distributed
```

In [ ]:
import logging
import os

from dask.distributed import Client

# distributed logs every worker registration and teardown at INFO,
# which is hundreds of lines per notebook and buries everything the
# diagnostic prints. Warnings and errors still come through.
for _name in ("distributed", "distributed.scheduler", "distributed.nanny",
              "distributed.worker", "distributed.core",
              "distributed.batched", "distributed.utils_perf"):
    logging.getLogger(_name).setLevel(logging.WARNING)

# Client.current() reuses a cluster if this cell is re-run, instead of
# quietly starting a second one alongside the first.
try:
    worker_count = max(1, int(os.environ.get("PBS_NCPUS", "1")))
except ValueError:
    worker_count = 1
    print("PBS_NCPUS is not an integer; using one worker")
try:
    client = Client.current()
except ValueError:
    try:
        # Separate worker processes: real memory isolation, and what
        # you want on ARE or any batch node.
        client = Client(n_workers=worker_count, threads_per_worker=1)
    except (RuntimeError, OSError) as error:
        # Some containers and locked-down environments cannot spawn
        # worker subprocesses. In-process workers still give a
        # distributed scheduler and a dashboard, just no isolation.
        print(f'process workers unavailable ({error}); '
              f'using in-process workers instead')
        client = Client(n_workers=worker_count, threads_per_worker=1,
                        processes=False)

active_workers = len(client.scheduler_info()['workers'])
print('dashboard:', client.dashboard_link)
print(f'workers: {active_workers} active (requested {worker_count})')
client

## Load the data

In [ ]:
zos = Dataset(short_name='zos', project='CMIP6', mip='Omon',
              exp='historical', dataset='ACCESS-CM2',
              ensemble='r1i1p1f1', grid='gn',
              timerange='2000/2009')

cube = regrid(zos.load(), target_grid='0.5x0.5', scheme='linear')
print(cube.summary(shorten=True))

## Helpers

In [ ]:
# The helpers below are lifted verbatim from
# cosima_cmip/cosima_common.py by notebooks/build_notebooks.py, so
# this notebook runs exactly the same code as the ESMValTool
# diagnostic script. Edit them there, then rebuild.
import contextlib
import warnings

try:  # TEOS-10, needed by the density helpers
    import gsw
except ImportError:
    gsw = None


EARTH_RADIUS = 6371000.0  # m, matches iris.analysis.cartography


FILL_VALUE_THRESHOLD = 1.0e19


def masked_data(cube, dtype=float):
    """Realised, fully masked data of a cube.

    Always go through this rather than ``cube.core_data()``.  The latter
    returns a *lazy* dask array, on which ``numpy.ma`` operations are
    silently no-ops: the mask is not applied, and fill values flow
    straight into the arithmetic.  This realises the data, applies any
    mask the file carries, and additionally masks non-finite values and
    unconverted fill values, which real CMOR files do contain.
    """
    data = np.ma.masked_invalid(np.ma.asarray(cube.data, dtype=dtype))
    return np.ma.masked_where(
        np.abs(np.ma.filled(data, 0.0)) >= FILL_VALUE_THRESHOLD, data)


def _coord_points(cube, name):
    """1D points for ``name``, taking column/row 0 of a 2D aux coord."""
    coord = cube.coord(name)
    if coord.ndim == 1:
        return coord.points
    points = coord.points
    # On a curvilinear grid latitude varies down a column and longitude
    # along a row; take the slice that actually varies.
    if name == "latitude":
        return points[:, 0]
    return points[0, :]


def lat_1d(cube):
    """Representative 1D latitude, also for 2D (curvilinear) coordinates."""
    return _coord_points(cube, "latitude")


def lon_1d(cube):
    """Representative 1D longitude, also for 2D coordinates."""
    return _coord_points(cube, "longitude")


def lat_2d(cube):
    """2D latitude field, broadcasting a 1D coordinate if needed."""
    coord = cube.coord("latitude")
    if coord.ndim == 2:
        return coord.points
    lon = cube.coord("longitude").points
    return np.broadcast_to(coord.points[:, None], (coord.shape[0], lon.size))


def lon_2d(cube):
    """2D longitude field, broadcasting a 1D coordinate if needed."""
    coord = cube.coord("longitude")
    if coord.ndim == 2:
        return coord.points
    lat = cube.coord("latitude").points
    return np.broadcast_to(coord.points[None, :], (lat.size, coord.shape[0]))


def cell_lengths(cube):
    """Zonal and meridional cell lengths in metres.

    Returns ``(dx, dy)`` broadcast to the horizontal shape of ``cube``.
    ``dx`` shrinks with the cosine of latitude, ``dy`` does not.
    """
    lat = lat_1d(cube)
    lon = lon_1d(cube)
    dlon = np.gradient(lon) * np.pi / 180.0
    dlat = np.gradient(lat) * np.pi / 180.0
    coslat = np.cos(np.deg2rad(lat))
    dx = EARTH_RADIUS * coslat[:, None] * dlon[None, :]
    dy = EARTH_RADIUS * np.broadcast_to(dlat[:, None], dx.shape)
    return dx, np.array(dy)


def horizontal_gradient(field, cube, periodic=True):
    """Central-difference gradient of ``field`` in m-1 units.

    ``field`` is a 2D masked array on the horizontal grid of ``cube``.
    Returns ``(d/dx, d/dy)``.  The zonal direction wraps when
    ``periodic`` is set, which is correct for a global CMOR grid.
    """
    dxm, dym = cell_lengths(cube)
    data = np.ma.masked_invalid(np.ma.asarray(field, dtype=float))
    filled = data.filled(np.nan)

    if periodic:
        padded = np.concatenate(
            [filled[:, -1:], filled, filled[:, :1]], axis=1)
        dfdx = (padded[:, 2:] - padded[:, :-2]) / (2.0 * dxm)
    else:
        dfdx = np.full_like(filled, np.nan)
        dfdx[:, 1:-1] = (filled[:, 2:] - filled[:, :-2]) / (2.0 * dxm[:, 1:-1])

    dfdy = np.full_like(filled, np.nan)
    dfdy[1:-1, :] = (filled[2:, :] - filled[:-2, :]) / (2.0 * dym[1:-1, :])

    return np.ma.masked_invalid(dfdx), np.ma.masked_invalid(dfdy)


def coriolis(lat):
    """Coriolis parameter in s-1."""
    if gsw is not None:
        return np.asarray(gsw.f(np.asarray(lat, dtype=float)))
    omega = 7.292115e-5
    return 2.0 * omega * np.sin(np.deg2rad(np.asarray(lat, dtype=float)))


def gravity(lat, depth=0.0):
    """Gravitational acceleration in m s-2."""
    if gsw is not None:
        return np.asarray(gsw.grav(np.asarray(lat, dtype=float), depth))
    return np.full(np.shape(lat), 9.7963)


def geostrophic_velocities(zos_field, cube, min_lat=5.0):
    """Surface geostrophic velocities from sea level.

    ``u = -(g/f) deta/dy`` and ``v = (g/f) deta/dx``.  Geostrophic
    balance is not valid close to the equator, where ``f`` vanishes.  The
    equatorial band through and including ``min_lat`` degrees either side
    of the equator is therefore masked (5 degrees by default).

    This finite-difference implementation requires the rectilinear grid
    produced by the recipe's ``regrid`` preprocessor.  Applying the
    one-dimensional spherical metrics to a native curvilinear MOM grid
    would silently use one arbitrary column of latitude and give a wrong
    derivative and equatorial mask.
    """
    if (cube.coord("latitude").ndim != 1
            or cube.coord("longitude").ndim != 1):
        raise ValueError(
            "geostrophic_velocities requires a rectilinear latitude/"
            "longitude grid; regrid zos before calculating derivatives")
    detadx, detady = horizontal_gradient(zos_field, cube)
    lat = lat_1d(cube)
    equator = np.abs(lat) <= min_lat
    # Mask f *before* dividing.  Masking the result afterwards still
    # evaluates 1/f on the equator, which can leak infinities or enormous
    # near-equatorial values into a reduction if a backend drops the mask.
    equator = np.broadcast_to(equator[:, None], detadx.shape)
    fcor = np.ma.masked_where(
        equator, np.broadcast_to(coriolis(lat)[:, None], detadx.shape))
    grav = np.broadcast_to(gravity(lat)[:, None], detadx.shape)
    u_g = -(grav / fcor) * detady
    v_g = (grav / fcor) * detadx
    return np.ma.masked_invalid(u_g), np.ma.masked_invalid(v_g)


def time_interval_days(cube, use_bounds=True):
    """Length of each time step in days, in the cube's own calendar.

    Do not convert a time coordinate to a hard-coded reference unit to
    do this.  ``cf_units`` refuses to convert between calendars, so
    ``coord.units.convert(points, "days since 1850-01-01")`` raises on
    any cube whose calendar is not ``standard`` -- and CMIP6 is full of
    them: ACCESS-CM2 is ``proleptic_gregorian``, UKESM1 and HadGEM3 are
    ``360_day``, several others are ``noleap``.

    Converting is also the wrong idea even where it works.  Month
    lengths are calendar-dependent, and they are exactly what a
    flux-weighted average needs: every month is 30 days in a
    ``360_day`` model and February never has 29 in a ``noleap`` one.
    Going through ``num2date`` lets the model's own calendar answer.

    Returns per-step lengths from the coordinate bounds, or the spacing
    between points where there are no bounds.
    """
    coord = cube.coord("time")
    if use_bounds and coord.has_bounds():
        dates = coord.units.num2date(coord.bounds)
        return np.array([(end - start).total_seconds() / 86400.0
                         for start, end in dates])

    dates = np.atleast_1d(coord.units.num2date(coord.points))
    if dates.size < 2:
        return np.array([30.0])
    spacing = np.array([(dates[i + 1] - dates[i]).total_seconds() / 86400.0
                        for i in range(dates.size - 1)])
    return np.concatenate([spacing, spacing[-1:]])


def time_step_days(cube):
    """Median spacing of the time coordinate, in days.

    Returns ``None`` when the cube has no usable time axis.
    """
    if not cube.coords("time") or cube.coord("time").shape[0] < 2:
        return None
    return float(np.median(time_interval_days(cube, use_bounds=False)))


def eddy_kinetic_energy(u_series, v_series):
    """Surface EKE from velocity time series, ``0.5*(u'^2 + v'^2)``."""
    u_anom = u_series - np.ma.mean(u_series, axis=0)
    v_anom = v_series - np.ma.mean(v_series, axis=0)
    return 0.5 * (np.ma.mean(u_anom**2, axis=0) + np.ma.mean(v_anom**2, axis=0))

## Compute

The equatorial band where $f$ vanishes is masked, two degrees
either side.

In [ ]:
data = masked_data(cube)
horizontal = next(cube.slices(['latitude', 'longitude']))

u_series, v_series = [], []
for step in range(data.shape[0]):
    u_field, v_field = geostrophic_velocities(
        data[step], horizontal, min_lat=2.0)
    u_series.append(u_field)
    v_series.append(v_field)
u_series = np.ma.stack(u_series)
v_series = np.ma.stack(v_series)

u_mean = np.ma.mean(u_series, axis=0)
v_mean = np.ma.mean(v_series, axis=0)
speed = np.ma.sqrt(u_mean**2 + v_mean**2)
variance = eddy_kinetic_energy(u_series, v_series)

step_days = time_step_days(cube)
if step_days is not None and step_days <= 7.0:
    variance_label = 'surface EKE'
else:
    variance_label = ('low-frequency surface KE\n'
                      '(monthly sea level: NOT mesoscale EKE)')
print(f'sampling interval: {step_days:.0f} days -> '
      f'{variance_label.splitlines()[0]}')

## Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 5))

mesh = axes[0].pcolormesh(lon_1d(horizontal), lat_1d(horizontal),
                          speed, cmap='viridis', shading='auto')
axes[0].set_title('Mean geostrophic speed')
fig.colorbar(mesh, ax=axes[0], shrink=0.8, label='m s$^{-1}$')

mesh = axes[1].pcolormesh(
    lon_1d(horizontal), lat_1d(horizontal),
    np.ma.log10(np.ma.masked_less_equal(variance, 0)),
    cmap='magma', shading='auto')
axes[1].set_title(variance_label)
fig.colorbar(mesh, ax=axes[1], shrink=0.8,
             label='log$_{10}$ (m$^2$ s$^{-2}$)')

for ax in axes:
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()

## Shut the cluster down

Workers hold their memory until the cluster closes. Leaving one running
in an idle kernel is the usual reason a later notebook cannot get the
memory it asks for.

In [ ]:
client.close()